In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import geopandas as gpd
import matplotlib.pyplot as plt

DATA = Path("data"); DATA.mkdir(exist_ok=True)

In [ ]:
"""
Step 1: Ontario FSA polygons and centre points

Source: StatCan 2021 FSA boundaries (simplified), [sachijay/canada_maps](https://github.com/sachijay/canada_maps).
Fixes the mislabelled CRS (actually EPSG:3347), keeps Ontario (K, L, M, N, P), and saves the
polygons plus one inside-the-polygon point per FSA. Expect 520 FSAs.
"""
SRC = ("https://raw.githubusercontent.com/sachijay/canada_maps/main/"
       "exported_files/forward_sortation_areas_simplified.geojson")

r = requests.get(SRC, timeout=120); r.raise_for_status()
if r.text.startswith("version https://git-lfs"):                 # Git LFS pointer → real file
    r = requests.get(SRC.replace("raw.githubusercontent.com",
                                 "media.githubusercontent.com/media"), timeout=120)
    r.raise_for_status()
(DATA / "fsa_canada_simplified.geojson").write_bytes(r.content)

fsa = gpd.read_file(DATA / "fsa_canada_simplified.geojson")
if abs(fsa.total_bounds).max() > 1000:                           # coordinates are metres
    fsa = fsa.set_crs(epsg=3347, allow_override=True)

on = fsa[fsa["CFSAUID"].str[0].isin(list("KLMNP"))][["CFSAUID", "geometry"]]
on = on.rename(columns={"CFSAUID": "FSA"}).reset_index(drop=True)
on["geometry"] = on.geometry.make_valid()

poly = on.to_crs(epsg=4326)
poly.to_file(DATA / "ontario_fsa_polygons.gpkg", driver="GPKG")

pts = on.copy()
pts["geometry"] = pts.geometry.representative_point()            # always inside the FSA
cent = pts.to_crs(epsg=4326)
cent["lon"], cent["lat"] = cent.geometry.x.round(4), cent.geometry.y.round(4)
cent.to_file(DATA / "ontario_fsa_centroids.gpkg", driver="GPKG")

print(len(poly), "FSAs | bounds:", poly.total_bounds.round(2))
print("points inside polygon:", cent.within(poly).sum(), "/", len(cent))

ax = poly.plot(figsize=(8, 8), edgecolor="grey", facecolor="none", linewidth=0.3)
cent.plot(ax=ax, markersize=2, color="red")
ax.set_title("Ontario FSAs and centre points"); ax.set_axis_off()

In [ ]:
"""
Copernicus setup

Sets the date range and connects to Copernicus (key from `~/.cdsapirc`).
`read_cds_csv` reads one downloaded file and converts temperature from Kelvin to °C.
"""

import cdsapi, zipfile

START = "2018-01-01"
END   = "2026-09-01"

client = cdsapi.Client()

def read_cds_csv(path):
    """Read a CDS time-series CSV (zipped or not) into time_utc + temp_c."""
    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:
            name = [n for n in z.namelist() if n.endswith(".csv")][0]
            df = pd.read_csv(z.open(name))
    else:
        df = pd.read_csv(path)
    tcol = [c for c in df.columns if "time" in c.lower()][0]
    vcol = [c for c in df.columns if c.lower() in ("t2m", "2m_temperature")][0]
    return pd.DataFrame({
        "time_utc": pd.to_datetime(df[tcol]),
        "temp_c": (df[vcol] - 273.15).astype("float32"),
    })

In [ ]:
"""
Download temperature per grid cell

Maps each FSA to its ERA5-Land grid cell (0.1°, ~9 km); FSAs in the same cell share one download.
Downloads each cell's full hourly series, 4 at a time, saving one file per cell. Re-run to resume or retry failures.
"""

from concurrent.futures import ThreadPoolExecutor, as_completed

CELLS = DATA / "era5land_cells"; CELLS.mkdir(exist_ok=True)

cent = gpd.read_file(DATA / "ontario_fsa_centroids.gpkg")
cent["cell_lat"] = cent["lat"].round(1)
cent["cell_lon"] = cent["lon"].round(1)
cent["cell"] = cent["cell_lat"].map("{:.1f}".format) + "_" + cent["cell_lon"].map("{:.1f}".format)
cent[["FSA", "cell", "cell_lat", "cell_lon"]].to_csv(DATA / "fsa_to_cell.csv", index=False)

cells = cent.drop_duplicates("cell")[["cell", "cell_lat", "cell_lon"]]
todo = cells[~cells["cell"].apply(lambda c: (CELLS / f"{c}.parquet").exists())]
todo = todo.head(8)   # TEST: only 8 cells — remove for the full run

print(f"{len(cent)} FSAs → {len(cells)} grid cells | {len(todo)} to download")

def get_cell(row):
    tmp = CELLS / f"{row.cell}.tmp"
    client.retrieve("reanalysis-era5-land-timeseries", {
        "variable": ["2m_temperature"],
        "location": {"longitude": float(row.cell_lon), "latitude": float(row.cell_lat)},
        "date": [f"{START}/{END}"],
        "data_format": "csv",
    }).download(str(tmp))
    read_cds_csv(tmp).to_parquet(CELLS / f"{row.cell}.parquet", index=False)
    tmp.unlink()
    return row.cell

done = 0
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(get_cell, r): r.cell for r in todo.itertuples()}
    for f in as_completed(futures):
        try:
            f.result(); done += 1
            print(f"{done}/{len(todo)} {futures[f]}")
        except Exception as e:
            print(f"FAILED {futures[f]}: {e}")    # re-run the cell later to retry

In [ ]:
"""
Combine into one FSA-level file

Joins each FSA to its cell's temperatures and adds fixed EST (UTC−5) time.
Saves `data/fsa_temperature_hourly.parquet` (~520 FSAs × ~76,000 hours).
"""
lookup = pd.read_csv(DATA / "fsa_to_cell.csv")
temps = pd.concat(
    [pd.read_parquet(f).assign(cell=f.stem) for f in sorted(CELLS.glob("*.parquet"))],
    ignore_index=True)

weather = lookup[["FSA", "cell"]].merge(temps, on="cell")
weather["time_est"] = weather["time_utc"] - pd.Timedelta(hours=5)    # fixed EST
weather["FSA"] = weather["FSA"].astype("category")
weather = weather[["FSA", "time_utc", "time_est", "temp_c"]]
weather.to_parquet(DATA / "fsa_temperature_hourly.parquet", index=False)

print(weather.shape, weather["FSA"].nunique(), "FSAs |", weather["temp_c"].isna().sum(), "missing")
print(weather["time_utc"].min(), "→", weather["time_utc"].max())

In [ ]:
weather = pd.read_parquet(DATA / "fsa_temperature_hourly.parquet")
weather.head(10)